In [1]:
# ==============================================================================
# CELL 1 REPLACEMENT: PRODUCTION-GRADE MONGODB VIDEO METADATA EXPLORATION
# ==============================================================================
# Comprehensive profiling covering: schema, null rates, categorical/numeric 
# distributions, array cardinality, cross-tabs, TEMPORAL FIELD PROFILING,
# REFERENTIAL INTEGRITY, BUSINESS RULE BASELINES, and anomaly detection baselines.
# ==============================================================================
from scripts.storage.config.paths import get_path_builder, SourceSystem
from scripts.storage.config.spark import get_spark_session
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json

# ------------------------------------------------------------------------------
# SESSION & LOAD
# ------------------------------------------------------------------------------
spark=spark = get_spark_session(app_name="mongo_silver_eda")
spark.sparkContext.setLogLevel("WARN")
path_builder = get_path_builder()

try:
    import dbutils  # type: ignore
    ingestion_date = dbutils.widgets.get("ingestion_date")
except Exception:
    ingestion_date = "2026-11-15"

print(f"Spark Session: {spark.sparkContext.appName}")
print(f"Ingestion Date: {ingestion_date}")

mongo_bronze_path = path_builder.bronze_partitioned(
    source=SourceSystem.MONGO,
    ingestion_date=ingestion_date,
    table="videos"
)
print(f"Bronze Path: {mongo_bronze_path}")

df = spark.read.parquet(mongo_bronze_path)
total_videos = df.count()
print(f"Loaded videos: {total_videos:,} rows x {len(df.columns)} cols")

# ------------------------------------------------------------------------------
# SCHEMA INSPECTION (FULL)
# ------------------------------------------------------------------------------
print("=" * 80)
print("SCHEMA — MONGODB VIDEOS (FULL)")
print("=" * 80)
df.printSchema()

# Nested struct detection
struct_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StructType)]
print(f"\nStruct columns detected: {struct_cols}")

# Flatten nested structs for downstream profiling
df_flat = df
for struct_col in struct_cols:
    nested_fields = [f.name for f in df.schema[struct_col].dataType.fields]
    for nf in nested_fields:
        df_flat = df_flat.withColumn(f"{struct_col}.{nf}", F.col(f"{struct_col}.{nf}"))

# ------------------------------------------------------------------------------
# NULL-RATE PROFILING (ALL COLUMNS INCLUDING NESTED)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("NULL-RATE PROFILE (ALL FIELDS)")
print("=" * 80)

total_rows = df.count()
null_report = []
for col_name in df_flat.columns:
    null_count = df_flat.filter(F.col(col_name).isNull()).count()
    null_pct = (null_count / total_rows) * 100 if total_rows > 0 else 0
    null_report.append((col_name, null_count, null_pct))

null_df = spark.createDataFrame(null_report, ["column", "null_count", "null_pct"])
null_df.orderBy(F.desc("null_pct")).show(truncate=False, n=50)

# ------------------------------------------------------------------------------
# CATEGORICAL DISTRIBUTIONS (TOP-K)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("CATEGORICAL DISTRIBUTIONS (TOP 20)")
print("=" * 80)

cat_cols = [
    "content_type", "status", 
    "competition.name", "competition.id",
    "technical.max_resolution", "technical.codec"
]

for col in cat_cols:
    if col in df_flat.columns:
        print(f"\n--- {col} ---")
        df_flat.filter(F.col(col).isNotNull()) \
               .groupBy(col) \
               .count() \
               .orderBy(F.desc("count")) \
               .show(20, truncate=False)

# Array columns: distinct element frequencies
array_cols = ["languages", "tags", "competition.teams"]
for col in array_cols:
    if col in df_flat.columns:
        print(f"\n--- {col} (element frequency) ---")
        df_flat.select(F.explode(F.col(col)).alias("element")) \
               .groupBy("element") \
               .count() \
               .orderBy(F.desc("count")) \
               .show(20, truncate=False)

# ------------------------------------------------------------------------------
# NUMERIC DISTRIBUTIONS (SUMMARY STATISTICS)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("NUMERIC DISTRIBUTIONS (SUMMARY)")
print("=" * 80)

num_cols = [
    "duration_sec", "match_duration_sec", "latent_popularity",
    "stats.views", "stats.likes"
]

for col in num_cols:
    if col in df_flat.columns:
        print(f"\n--- {col} ---")
        df_flat.select(col).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show(truncate=False)

# ------------------------------------------------------------------------------
# ARRAY CARDINALITY PROFILES
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("ARRAY CARDINALITY PROFILES")
print("=" * 80)

for col in array_cols:
    if col in df_flat.columns:
        print(f"\n--- {col} size distribution ---")
        df_flat.select(F.size(F.col(col)).alias("size")) \
               .groupBy("size") \
               .count() \
               .orderBy("size") \
               .show(truncate=False)

# ------------------------------------------------------------------------------
# TEMPORAL FIELD PROFILING (CRITICAL GAP FROM ORIGINAL)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("TEMPORAL FIELD PROFILING — kickoff_at, live_start_at, live_end_at")
print("=" * 80)

temporal_cols = ["created_at", "last_updated", "kickoff_at", "live_start_at", "live_end_at"]

# Null rates & ranges
for col in temporal_cols:
    if col in df_flat.columns:
        null_cnt = df_flat.filter(F.col(col).isNull()).count()
        non_null = df_flat.filter(F.col(col).isNotNull())
        min_val = non_null.agg(F.min(col)).collect()[0][0]
        max_val = non_null.agg(F.max(col)).collect()[0][0]
        print(f"\n{col}:")
        print(f"  Null count: {null_cnt:,} ({(null_cnt/total_rows)*100:.2f}%)")
        print(f"  Range: {min_val} → {max_val}")

# Logical ordering validations
print("\n--- TEMPORAL LOGICAL ORDERING VALIDATIONS ---")

# 1. live_start_at < live_end_at (when both present)
if "live_start_at" in df_flat.columns and "live_end_at" in df_flat.columns:
    bad_order = df_flat.filter(
        F.col("live_start_at").isNotNull() & 
        F.col("live_end_at").isNotNull() & 
        (F.col("live_start_at") >= F.col("live_end_at"))
    ).count()
    print(f"live_start_at >= live_end_at (invalid): {bad_order:,}")

# 2. kickoff_at alignment with live_start_at (when both present, should be close)
if "kickoff_at" in df_flat.columns and "live_start_at" in df_flat.columns:
    aligned = df_flat.filter(
        F.col("kickoff_at").isNotNull() & 
        F.col("live_start_at").isNotNull()
    )
    # Within 5 minutes tolerance
    within_5min = aligned.filter(
        F.abs(F.unix_timestamp("kickoff_at") - F.unix_timestamp("live_start_at")) <= 300
    ).count()
    total_both = aligned.count()
    print(f"kickoff_at vs live_start_at: {within_5min:,}/{total_both:,} within 5min tolerance")
    if total_both > within_5min:
        outliers = aligned.filter(
            F.abs(F.unix_timestamp("kickoff_at") - F.unix_timestamp("live_start_at")) > 300
        ).select("kickoff_at", "live_start_at").limit(5)
        print(f"  Outlier examples:")
        outliers.show(truncate=False)

# 3. created_at <= last_updated (when both present)
if "created_at" in df_flat.columns and "last_updated" in df_flat.columns:
    bad_created = df_flat.filter(
        F.col("created_at").isNotNull() & 
        F.col("last_updated").isNotNull() & 
        (F.col("created_at") > F.col("last_updated"))
    ).count()
    print(f"created_at > last_updated (invalid): {bad_created:,}")

# 4. Temporal fields by content_type
print("\n--- TEMPORAL FIELD PRESENCE BY content_type ---")
for ct in ["live", "highlight", "replay"]:
    if "content_type" in df_flat.columns:
        subset = df_flat.filter(F.col("content_type") == ct)
        cnt = subset.count()
        if cnt > 0:
            for tc in temporal_cols:
                if tc in df_flat.columns:
                    present = subset.filter(F.col(tc).isNotNull()).count()
                    print(f"  {ct}: {tc} present={present:,}/{cnt:,} ({(present/cnt)*100:.1f}%)")

# ------------------------------------------------------------------------------
# REFERENTIAL INTEGRITY CHECKS (CRITICAL GAP FROM ORIGINAL)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("REFERENTIAL INTEGRITY — live content mandatory fields")
print("=" * 80)

if "content_type" in df_flat.columns:
    live_df = df_flat.filter(F.col("content_type") == "live")
    live_count = live_df.count()
    print(f"\nTotal live videos: {live_count:,}")
    
    if live_count > 0:
        # match_id mandatory for live
        missing_match_id = live_df.filter(
            F.col("match_id").isNull() | (F.trim(F.col("match_id")) == "")
        ).count()
        print(f"  MISSING match_id: {missing_match_id:,} ({(missing_match_id/live_count)*100:.2f}%)")
        
        # competition.id mandatory for live
        missing_comp_id = live_df.filter(
            F.col("competition.id").isNull() | (F.trim(F.col("competition.id")) == "")
        ).count()
        print(f"  MISSING competition.id: {missing_comp_id:,} ({(missing_comp_id/live_count)*100:.2f}%)")
        
        # live_start_at mandatory for live
        missing_live_start = live_df.filter(F.col("live_start_at").isNull()).count()
        print(f"  MISSING live_start_at: {missing_live_start:,} ({(missing_live_start/live_count)*100:.2f}%)")
        
        # live_end_at mandatory for live
        missing_live_end = live_df.filter(F.col("live_end_at").isNull()).count()
        print(f"  MISSING live_end_at: {missing_live_end:,} ({(missing_live_end/live_count)*100:.2f}%)")
        
        # kickoff_at expected for live
        missing_kickoff = live_df.filter(F.col("kickoff_at").isNull()).count()
        print(f"  MISSING kickoff_at: {missing_kickoff:,} ({(missing_kickoff/live_count)*100:.2f}%)")

# Also check: competition.name presence when competition.id present
if "competition.id" in df_flat.columns and "competition.name" in df_flat.columns:
    comp_id_present = df_flat.filter(
        F.col("competition.id").isNotNull() & (F.trim(F.col("competition.id")) != "")
    )
    missing_comp_name = comp_id_present.filter(
        F.col("competition.name").isNull() | (F.trim(F.col("competition.name")) == "")
    ).count()
    total_comp_id = comp_id_present.count()
    print(f"\ncompetition.id present but competition.name missing: {missing_comp_name:,}/{total_comp_id:,}")

# ------------------------------------------------------------------------------
# BUSINESS RULE BASELINE VALIDATION (CRITICAL GAP FROM ORIGINAL)
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("BUSINESS RULE BASELINES — duration_sec bounds per content_type")
print("=" * 80)

# Expected bounds from VideoSimulation.WATCH_TIME_BOUNDS (R18)
# live: typically full match duration (90 min + extra)
# highlight: shorter clips (1-10 min)
# replay: full match duration
EXPECTED_BOUNDS = {
    "live": (300, 7200),      # 5 min - 2 hours (match window)
    "highlight": (10, 1200),  # 10 sec - 20 min
    "replay": (300, 10800)    # 5 min - 3 hours
}

if "content_type" in df_flat.columns and "duration_sec" in df_flat.columns:
    for ct, (exp_min, exp_max) in EXPECTED_BOUNDS.items():
        subset = df_flat.filter(F.col("content_type") == ct)
        cnt = subset.count()
        if cnt > 0:
            # Null duration
            null_dur = subset.filter(F.col("duration_sec").isNull()).count()
            # Below expected min
            below_min = subset.filter(F.col("duration_sec") < exp_min).count()
            # Above expected max
            above_max = subset.filter(F.col("duration_sec") > exp_max).count()
            # Zero or negative
            zero_or_neg = subset.filter(F.col("duration_sec") <= 0).count()
            
            print(f"\n{ct.upper()} (expected {exp_min}s–{exp_max}s):")
            print(f"  Count: {cnt:,}")
            print(f"  Null duration_sec: {null_dur:,} ({(null_dur/cnt)*100:.2f}%)")
            print(f"  ≤0 (invalid): {zero_or_neg:,} ({(zero_or_neg/cnt)*100:.2f}%)")
            print(f"  < {exp_min}s (below expected): {below_min:,} ({(below_min/cnt)*100:.2f}%)")
            print(f"  > {exp_max}s (above expected): {above_max:,} ({(above_max/cnt)*100:.2f}%)")
            
            # Show distribution
            subset.filter(F.col("duration_sec").isNotNull()) \
                  .select("duration_sec") \
                  .summary("min", "25%", "50%", "75%", "max") \
                  .show(truncate=False)

# match_duration_sec validation for live content
if "content_type" in df_flat.columns and "match_duration_sec" in df_flat.columns:
    live_subset = df_flat.filter(F.col("content_type") == "live")
    live_cnt = live_subset.count()
    if live_cnt > 0:
        null_md = live_subset.filter(F.col("match_duration_sec").isNull()).count()
        zero_md = live_subset.filter(F.col("match_duration_sec") <= 0).count()
        print(f"\nmatch_duration_sec for live content:")
        print(f"  Null: {null_md:,} ({(null_md/live_cnt)*100:.2f}%)")
        print(f"  ≤0: {zero_md:,} ({(zero_md/live_cnt)*100:.2f}%)")
        live_subset.filter(F.col("match_duration_sec").isNotNull()) \
                   .select("match_duration_sec") \
                   .summary("min", "25%", "50%", "75%", "max") \
                   .show(truncate=False)

# ------------------------------------------------------------------------------
# CROSS-TAB & COMPOSITE VALIDATIONS
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("CROSS-TAB & COMPOSITE VALIDATIONS")
print("=" * 80)

# content_type x status
if "content_type" in df_flat.columns and "status" in df_flat.columns:
    print("\nCross-tab: content_type × status")
    df_flat.groupBy("content_type", "status").count() \
           .orderBy("content_type", F.desc("count")) \
           .show(truncate=False)

# stats.likes > stats.views (invariant break baseline)
if "stats.views" in df_flat.columns and "stats.likes" in df_flat.columns:
    likes_gt_views = df_flat.filter(
        F.col("stats.likes").isNotNull() & 
        F.col("stats.views").isNotNull() & 
        (F.col("stats.likes") > F.col("stats.views"))
    ).count()
    neg_views = df_flat.filter(F.col("stats.views") < 0).count()
    neg_likes = df_flat.filter(F.col("stats.likes") < 0).count()
    print(f"\n[Baseline] likes > views: {likes_gt_views:,}")
    print(f"[Baseline] Negative views: {neg_views:,}, Negative likes: {neg_likes:,}")
    
    # Stats distribution
    stats_summary = df_flat.select("stats.views", "stats.likes").agg(
        F.min("views").alias("min_views"), F.max("views").alias("max_views"),
        F.min("likes").alias("min_likes"), F.max("likes").alias("max_likes"),
        F.mean("views").alias("mean_views"), F.mean("likes").alias("mean_likes")
    ).collect()[0]
    print(f"  Views: min={stats_summary['min_views']}, max={stats_summary['max_views']}, mean={stats_summary['mean_views']:.1f}")
    print(f"  Likes: min={stats_summary['min_likes']}, max={stats_summary['max_likes']}, mean={stats_summary['mean_likes']:.1f}")

# competition metadata completeness
if "competition.id" in df_flat.columns:
    empty_comp_id = df_flat.filter(
        F.col("competition.id").isNull() | (F.trim(F.col("competition.id")) == "")
    ).count()
    print(f"\n[Baseline] Empty competition.id: {empty_comp_id:,}")

if "tags" in df_flat.columns:
    empty_tags = df_flat.filter(F.size(F.col("tags")) == 0).count()
    print(f"[Baseline] Empty tags array: {empty_tags:,}")

# ------------------------------------------------------------------------------
# SUMMARY STATISTICS FOR ANOMALY RATE ESTIMATION
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SUMMARY: CLEAN BASELINE METRICS FOR ANOMALY DETECTION")
print("=" * 80)

total_videos = df.count()
print(f"Total videos: {total_videos:,}")
print(f"Expected anomaly injection rate: ~3.5% blended across 8 types")
print(f"Expected per anomaly type: ~{total_videos * 0.035 / 8:,.0f} rows")
print(f"Total expected dirty rows: ~{total_videos * 0.035:,.0f} (3.5%)")

# Key baseline flags for transformer DQ design
print("\n--- KEY BASELINE FLAGS FOR SILVER TRANSFORMER DQ DESIGN ---")
baseline_flags = {
    "missing_title": df.filter(F.col("title").isNull() | (F.trim(F.col("title")) == "")).count(),
    "invalid_duration_le_zero": df.filter(F.col("duration_sec").isNotNull() & (F.col("duration_sec") <= 0)).count(),
    "invalid_duration_excessive": df.filter(F.col("duration_sec") > 20000).count(),
    "invalid_content_type": df.filter(~F.col("content_type").isin("live", "highlight", "replay")).count(),
    "live_missing_match_id": df.filter((F.col("content_type") == "live") & (F.col("match_id").isNull() | (F.trim(F.col("match_id")) == ""))).count(),
    "live_status_contradiction": df.filter((F.col("content_type") == "live") & (F.col("status") == "ended") & F.col("live_start_at").isNotNull() & (F.col("live_start_at") > F.current_timestamp())).count(),
    "likes_gt_views": df.filter(F.col("stats.likes").isNotNull() & F.col("stats.views").isNotNull() & (F.col("stats.likes") > F.col("stats.views"))).count(),
    "empty_competition": df.filter(F.col("competition.id").isNull() | (F.trim(F.col("competition.id")) == "")).count(),
    "empty_tags": df.filter(F.size(F.col("tags")) == 0).count(),
}

for flag, count in baseline_flags.items():
    print(f"  {flag}: {count:,} ({(count/total_videos)*100:.4f}%)")

print("\n" + "=" * 80)
print("EXPLORATION COMPLETE — Ready for Silver Transformer Design")
print("=" * 80)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 12:25:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session: mongo_silver_eda
Ingestion Date: 2026-11-15
Bronze Path: s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_date=2026-11-15/videos.parquet


26/11/15 12:25:48 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Loaded videos: 10,000 rows x 20 cols
SCHEMA — MONGODB VIDEOS (FULL)
root
 |-- _id: string (nullable = true)
 |-- competition: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- teams: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- content_type: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- description: string (nullable = true)
 |-- duration_sec: integer (nullable = true)
 |-- kickoff_at: timestamp (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- last_updated: timestamp (nullable = true)
 |-- latent_popularity: integer (nullable = true)
 |-- live_end_at: timestamp (nullable = true)
 |-- live_start_at: timestamp (nullable = true)
 |-- match_duration_sec: integer (nullable = true)
 |-- match_id: string (nullable = true)
 |-- stats: struct (nullable = true)
 |    |-- views: integer (nullable = true)
 | 

+------------------------+----------+-------------------+
|column                  |null_count|null_pct           |
+------------------------+----------+-------------------+
|live_end_at             |6683      |66.83              |
|live_start_at           |6683      |66.83              |
|match_id                |49        |0.49               |
|title                   |47        |0.47000000000000003|
|competition.name        |47        |0.47000000000000003|
|competition.id          |47        |0.47000000000000003|
|competition.teams       |47        |0.47000000000000003|
|video_id                |0         |0.0                |
|status                  |0         |0.0                |
|stats.views             |0         |0.0                |
|match_duration_sec      |0         |0.0                |
|stats.likes             |0         |0.0                |
|kickoff_at              |0         |0.0                |
|technical.max_resolution|0         |0.0                |
|stats        

+-------+------------------+
|summary|duration_sec      |
+-------+------------------+
|count  |10000             |
|mean   |4682.6593         |
|stddev |3182.1672817569406|
|min    |-95               |
|25%    |497               |
|50%    |6480              |
|75%    |6900              |
|max    |20000             |
+-------+------------------+


--- match_duration_sec ---
+-------+------------------+
|summary|match_duration_sec|
+-------+------------------+
|count  |10000             |
|mean   |6748.698          |
|stddev |276.94442514241337|
|min    |6300              |
|25%    |6480              |
|50%    |6780              |
|75%    |7020              |
|max    |7200              |
+-------+------------------+


--- latent_popularity ---
+-------+-----------------+
|summary|latent_popularity|
+-------+-----------------+
|count  |10000            |
|mean   |48.1182          |
|stddev |44.54879895472864|
|min    |3                |
|25%    |17               |
|50%    |29            

In [2]:
from scripts.storage.config.paths import get_path_builder, SourceSystem
from scripts.storage.config.spark import get_spark_session
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import ArrayType, StructType, NumericType, StringType

spark = get_spark_session(app_name="mongo_silver_eda")
path_builder = get_path_builder()

try:
    import dbutils  # type: ignore
    ingestion_date = dbutils.widgets.get("ingestion_date")
except Exception:
    ingestion_date = "2026-11-15"

print(f"Spark Session: {spark.sparkContext.appName}")
print(f"Ingestion Date: {ingestion_date}")

mongo_bronze_path = path_builder.bronze_partitioned(
    source=SourceSystem.MONGO,
    ingestion_date=ingestion_date,
    table="videos"
)
print(f"Bronze Path: {mongo_bronze_path}")
df = spark.read.parquet(mongo_bronze_path)
# =============================================================================
# SUPPLEMENTARY: DIRTY DATA INJECTION PROFILING — MONGODB VIDEOS
# Targets anomalies from MongoDirtyDataInjector._inject_video_anomalies()
# =============================================================================

print("=" * 80)
print("DIRTY DATA INJECTION PROFILING — MONGODB VIDEOS")
print("=" * 80)

# ---------------------------------------------------------------
# Anomaly Type 1: Null mandatory field — title
# ---------------------------------------------------------------
null_title = df.filter(F.col("title").isNull()).count()
empty_title = df.filter(F.trim(F.col("title")) == "").count()
print(f"\n[Anomaly 1] Null title: {null_title:,}, Empty title: {empty_title:,}")

# ---------------------------------------------------------------
# Anomaly Type 2 & 3: Invalid duration_sec — zero/negative OR excessively large
# Type 2: random.randint(-100, 0)  -> zero or negative
# Type 3: 20000 (exceeds match window)
# ---------------------------------------------------------------
neg_or_zero_dur = df.filter(F.col("duration_sec").isNotNull() & (F.col("duration_sec") <= 0)).count()
extreme_dur = df.filter(F.col("duration_sec") > 10800).count()  # > 3 hours
print(f"\n[Anomaly 2] Zero/negative duration_sec: {neg_or_zero_dur:,}")
print(f"[Anomaly 3] Excessive duration_sec (> 10800s): {extreme_dur:,}")

# Full duration distribution
print("\nDuration distribution (including dirty):")
dur_stats = df.filter(F.col("duration_sec").isNotNull()).agg(
    F.min("duration_sec").alias("min"),
    F.max("duration_sec").alias("max"),
    F.mean("duration_sec").alias("mean"),
    F.expr("percentile_approx(duration_sec, 0.5)").alias("median"),
    F.expr("percentile_approx(duration_sec, 0.95)").alias("p95"),
    F.expr("percentile_approx(duration_sec, 0.99)").alias("p99")
).collect()[0]
print(f"  Min: {dur_stats['min']}, Max: {dur_stats['max']}, Mean: {dur_stats['mean']:.1f}")
print(f"  Median: {dur_stats['median']}, P95: {dur_stats['p95']}, P99: {dur_stats['p99']}")

# Show extreme values
if extreme_dur > 0:
    df.filter(F.col("duration_sec") > 10800) \
        .select("video_id", "content_type", "duration_sec") \
        .show(20, truncate=False)

# ---------------------------------------------------------------
# Anomaly Type 4: Invalid content_type enum ("INVALID_TYPE")
# ---------------------------------------------------------------
invalid_content_type = df.filter(F.col("content_type") == "INVALID_TYPE").count()
print(f"\n[Anomaly 4] Invalid content_type='INVALID_TYPE': {invalid_content_type:,}")

print("\nFull content_type distribution (including dirty):")
df.groupBy("content_type").count().orderBy(F.desc("count")).show(truncate=False)

# ---------------------------------------------------------------
# Anomaly Type 5: Referential orphan — live content with missing match_id
# Sets: content_type="live", match_id=None, live_start_at=None, live_end_at=None
# ---------------------------------------------------------------
live_no_match = df.filter(
    (F.col("content_type") == "live") & 
    (F.col("match_id").isNull() | (F.trim(F.col("match_id")) == ""))
).count()
print(f"\n[Anomaly 5] Live videos without match_id: {live_no_match:,}")

# Also check for missing live_start_at / live_end_at on live content
live_missing_window = df.filter(
    (F.col("content_type") == "live") & 
    (F.col("live_start_at").isNull() | F.col("live_end_at").isNull())
).count()
print(f"[Anomaly 5] Live videos missing live_start_at/live_end_at: {live_missing_window:,}")

# ---------------------------------------------------------------
# Anomaly Type 6: Status contradiction — live content with "ended" status
# ---------------------------------------------------------------
live_ended_status = df.filter(
    (F.col("content_type") == "live") & (F.col("status") == "ended")
).count()
print(f"\n[Anomaly 6] Live videos with status='ended': {live_ended_status:,}")

print("\nFull status distribution (including dirty):")
df.groupBy("status").count().orderBy(F.desc("count")).show(truncate=False)

# Cross-tab: content_type vs status
print("\nCross-tab: content_type x status")
df.groupBy("content_type", "status").count().orderBy("content_type", F.desc("count")).show(truncate=False)

# ---------------------------------------------------------------
# Anomaly Type 7: Metric invariant break — likes > views
# Sets: views=150, likes=random(151, 500)
# ---------------------------------------------------------------
likes_gt_views = df.filter(
    F.col("stats.likes").isNotNull() & 
    F.col("stats.views").isNotNull() & 
    (F.col("stats.likes") > F.col("stats.views"))
).count()
print(f"\n[Anomaly 7] Videos with likes > views: {likes_gt_views:,}")

# Also check for negative stats (not injected but good to verify)
neg_views = df.filter(F.col("stats.views") < 0).count()
neg_likes = df.filter(F.col("stats.likes") < 0).count()
print(f"  Negative views: {neg_views:,}, Negative likes: {neg_likes:,}")

# Stats distribution
if "stats" in df.columns:
    stats_df = df.select("stats.views", "stats.likes")
    stats_summary = stats_df.agg(
        F.min("views").alias("min_views"), F.max("views").alias("max_views"),
        F.min("likes").alias("min_likes"), F.max("likes").alias("max_likes"),
        F.mean("views").alias("mean_views"), F.mean("likes").alias("mean_likes")
    ).collect()[0]
    print(f"  Views: min={stats_summary['min_views']}, max={stats_summary['max_views']}, mean={stats_summary['mean_views']:.1f}")
    print(f"  Likes: min={stats_summary['min_likes']}, max={stats_summary['max_likes']}, mean={stats_summary['mean_likes']:.1f}")

# ---------------------------------------------------------------
# Anomaly Type 8: Missing competition metadata
# Sets: competition={}, tags=[]
# ---------------------------------------------------------------
# Check for empty competition struct
if "competition" in df.columns:
    empty_competition = df.filter(
        F.col("competition").isNotNull() & 
        (F.col("competition.id").isNull() | (F.trim(F.col("competition.id")) == ""))
    ).count()
    print(f"\n[Anomaly 8] Videos with empty competition.id: {empty_competition:,}")

# Check for empty tags array
if "tags" in df.columns:
    empty_tags = df.filter(F.size(F.col("tags")) == 0).count()
    print(f"[Anomaly 8] Videos with empty tags array: {empty_tags:,}")

# ---------------------------------------------------------------
# SUMMARY: Blended anomaly rate estimation
# ---------------------------------------------------------------
total_videos = df.count()
print(f"\n{'=' * 80}")
print(f"ESTIMATED ANOMALY COUNTS (at ~3.5% injection rate, 8 types):")
print(f"  Expected per anomaly type: ~{total_videos * 0.035 / 8:,.0f} rows")
print(f"  Total expected dirty rows: ~{total_videos * 0.035:,.0f} ({3.5}%)")
print(f"{'=' * 80}")


Spark Session: mongo_silver_eda
Ingestion Date: 2026-11-15
Bronze Path: s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_date=2026-11-15/videos.parquet
DIRTY DATA INJECTION PROFILING — MONGODB VIDEOS

[Anomaly 1] Null title: 47, Empty title: 0

[Anomaly 2] Zero/negative duration_sec: 39
[Anomaly 3] Excessive duration_sec (> 10800s): 46

Duration distribution (including dirty):
  Min: -95, Max: 20000, Mean: 4682.7
  Median: 6480, P95: 7140, P99: 7200
+------------------------------------+------------+------------+
|video_id                            |content_type|duration_sec|
+------------------------------------+------------+------------+
|01a1dedc-022a-7ece-bb83-cce0810a76bc|live        |20000       |
|01a1dedc-02fd-7be5-9c36-9b1067fd3c68|highlight   |20000       |
|01a1dedc-0321-7e0a-a36a-cad63e20017a|highlight   |20000       |
|01a1dedc-0345-7b74-b9d8-175c097b4d2f|highlight   |20000       |
|01a1dedc-0354-7743-95ed-269c3974d24f|highlight   |20000       |
|01a1dedc-0